In [2]:
# --- IMPORTS ---
import numpy as np
import pandas as pd
import json, re, html, os, pickle

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer
from sklearn.metrics import f1_score

os.makedirs('./checkpoints', exist_ok=True)
print("✓ Libraries loaded")
print(f"  Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")


✓ Libraries loaded
  Device: cpu


In [ ]:
# --- SETTINGS ---
MAX_LEN       = 64
BATCH_SIZE_RNN  = 32
BATCH_SIZE_BERT = 16
EMBEDDING_DIM   = 100
GLOVE_PATH      = './data/glove/glove.6B.100d.txt'

TARGET_LABELS = [
    'ineffective', 'unnecessary', 'pharma', 'rushed', 'side-effect',
    'mandatory', 'country', 'ingredients', 'political', 'none',
    'conspiracy', 'religious'
]
print(f"Labels ({len(TARGET_LABELS)}): {TARGET_LABELS}")

Labels (12): ['ineffective', 'unnecessary', 'pharma', 'rushed', 'side-effect', 'mandatory', 'country', 'ingredients', 'political', 'none', 'conspiracy', 'religious']


In [4]:
# --- DATA LOADING & CLEANING ---
def load_data(file_path, is_test=False):
    with open(file_path, 'r') as f:
        data = json.load(f)
    rows = []
    for item in data:
        row = {'ID': item['ID'], 'tweet': item['tweet']}
        if not is_test:
            for label in TARGET_LABELS:
                row[label] = 1 if label in item['labels'] else 0
        rows.append(row)
    return pd.DataFrame(rows)

def clean_tweet(text):
    text = html.unescape(text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)  # remove URLs
    text = re.sub(r'@\w+', '', text)                    # remove @mentions
    text = re.sub(r'#(\w+)', r'\1', text)               # strip # from hashtags
    return ' '.join(text.split()).lower()

train_df = load_data('./data/train.json')
val_df   = load_data('./data/val.json')
test_df  = load_data('./data/test.json', is_test=True)

for df in [train_df, val_df, test_df]:
    df['tweet'] = df['tweet'].apply(clean_tweet)

print(f"✓ Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

✓ Train: 6956 | Val: 987 | Test: 1976


In [5]:
# --- EDA: LABEL DISTRIBUTION ---
print(f"\n{'Label':<15} {'Count':>6} {'%':>7}  {'Raw pos_weight':>15}  {'Capped weight':>15}")
print("-" * 65)
for label in TARGET_LABELS:
    count  = train_df[label].sum()
    pct    = count / len(train_df) * 100
    neg    = (train_df[label] == 0).sum()
    raw_w  = neg / (count + 1e-8)
    cap_w  = min(raw_w, 10.0)
    print(f"{label:<15} {count:>6} {pct:>6.1f}%  {raw_w:>15.2f}  {cap_w:>15.2f}")


Label            Count       %   Raw pos_weight    Capped weight
-----------------------------------------------------------------
ineffective       1171   16.8%             4.94             4.94
unnecessary        503    7.2%            12.83            10.00
pharma             889   12.8%             6.82             6.82
rushed            1031   14.8%             5.75             5.75
side-effect       2663   38.3%             1.61             1.61
mandatory          548    7.9%            11.69            10.00
country            140    2.0%            48.69            10.00
ingredients        304    4.4%            21.88            10.00
political          437    6.3%            14.92            10.00
none               440    6.3%            14.81            10.00
conspiracy         341    4.9%            19.40            10.00
religious           45    0.6%           153.58            10.00


In [6]:
# --- CLASS IMBALANCE: POS_WEIGHTS (capped at 10.0) ---
pos_weights = []
for label in TARGET_LABELS:
    pos  = (train_df[label] == 1).sum()
    neg  = (train_df[label] == 0).sum()
    w    = min(neg / (pos + 1e-8), 10.0)
    pos_weights.append(w)
pos_weights = torch.tensor(pos_weights, dtype=torch.float32)
print(f"✓ pos_weights computed and capped at 10.0")
print(f"  {pos_weights.tolist()}")

✓ pos_weights computed and capped at 10.0
  [4.940222263336182, 10.0, 6.824522018432617, 5.746847629547119, 1.6120916604995728, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0]


In [7]:
# --- GLOVE EMBEDDINGS & VOCABULARY ---
def load_glove(path, dim=100):
    glove = {}
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.split()
            glove[parts[0]] = np.array(parts[1:], dtype=np.float32)
    print(f"✓ Loaded {len(glove):,} GloVe vectors")
    return glove

def build_vocab_and_matrix(df_list, glove, dim=100):
    all_words = set()
    for df in df_list:
        for tweet in df['tweet']:
            all_words.update(tweet.split())
    
    word2idx = {'<PAD>': 0, '<UNK>': 1}
    for w in sorted(all_words):
        word2idx[w] = len(word2idx)
    
    matrix = np.random.uniform(-0.1, 0.1, (len(word2idx), dim)).astype(np.float32)
    matrix[0] = 0.0  # PAD = zeros
    
    hits = sum(1 for w in word2idx if w in glove)
    for w, i in word2idx.items():
        if w in glove:
            matrix[i] = glove[w]
    
    print(f"✓ Vocab size: {len(word2idx):,}")
    print(f"✓ GloVe coverage: {hits}/{len(word2idx)} ({hits/len(word2idx)*100:.1f}%)")
    return word2idx, torch.tensor(matrix, dtype=torch.float32)

glove_vecs = load_glove(GLOVE_PATH, EMBEDDING_DIM)
word2idx, embed_matrix = build_vocab_and_matrix(
    [train_df, val_df, test_df], glove_vecs, EMBEDDING_DIM
)


✓ Loaded 400,000 GloVe vectors
✓ Vocab size: 31,756
✓ GloVe coverage: 12529/31756 (39.5%)


In [8]:
# --- DATASET CLASS ---
class TweetDataset(Dataset):
    def __init__(self, df, tokenizer, target_labels, max_len=64,
                 model_type='bert', word2idx=None):
        self.df           = df
        self.tokenizer    = tokenizer
        self.target_labels= target_labels
        self.max_len      = max_len
        self.model_type   = model_type
        self.word2idx     = word2idx
        self.has_labels   = all(l in df.columns for l in target_labels)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        tweet = row['tweet']

        if self.model_type == 'bert':
            enc = self.tokenizer(tweet, max_length=self.max_len,
                                 padding='max_length', truncation=True,
                                 return_tensors='pt')
            out = {
                'input_ids':      enc['input_ids'].squeeze(),
                'attention_mask': enc['attention_mask'].squeeze(),
                'token_type_ids': enc.get('token_type_ids', torch.zeros(self.max_len, dtype=torch.long)).squeeze(),
            }
        else:  # rnn
            tokens = tweet.split()[:self.max_len]
            ids    = [self.word2idx.get(t, self.word2idx['<UNK>']) for t in tokens]
            pad    = self.word2idx['<PAD>']
            ids   += [pad] * (self.max_len - len(ids))
            out    = {
                'input_ids':      torch.tensor(ids, dtype=torch.long),
                'attention_mask': torch.tensor([1]*len(tokens) + [0]*(self.max_len - len(tokens)), dtype=torch.long),
            }

        if self.has_labels:
            out['labels'] = torch.tensor([row[l] for l in self.target_labels], dtype=torch.float32)
        return out

bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
print("✓ TweetDataset class ready")


✓ TweetDataset class ready


In [9]:
# --- SAVE ALL ARTIFACTS TO ./checkpoints/ ---
train_df.to_pickle('./checkpoints/train_df.pkl')
val_df.to_pickle('./checkpoints/val_df.pkl')
test_df.to_pickle('./checkpoints/test_df.pkl')

with open('./checkpoints/word2idx.pkl', 'wb') as f:
    pickle.dump(word2idx, f)

torch.save(embed_matrix,  './checkpoints/embed_matrix.pt')
torch.save(pos_weights,   './checkpoints/pos_weights.pt')

# Also pickle the Dataset class and settings for reuse
settings = {
    'MAX_LEN': MAX_LEN,
    'BATCH_SIZE_RNN': BATCH_SIZE_RNN,
    'BATCH_SIZE_BERT': BATCH_SIZE_BERT,
    'EMBEDDING_DIM': EMBEDDING_DIM,
    'TARGET_LABELS': TARGET_LABELS,
}
with open('./checkpoints/settings.pkl', 'wb') as f:
    pickle.dump(settings, f)

print("✓ Saved to ./checkpoints/:")
print(f"  train_df.pkl     ({len(train_df)} rows)")
print(f"  val_df.pkl       ({len(val_df)} rows)")
print(f"  test_df.pkl      ({len(test_df)} rows)")
print(f"  word2idx.pkl     ({len(word2idx):,} words)")
print(f"  embed_matrix.pt  {tuple(embed_matrix.shape)}")
print(f"  pos_weights.pt   {pos_weights.tolist()}")
print(f"  settings.pkl")


✓ Saved to ./checkpoints/:
  train_df.pkl     (6956 rows)
  val_df.pkl       (987 rows)
  test_df.pkl      (1976 rows)
  word2idx.pkl     (31,756 words)
  embed_matrix.pt  (31756, 100)
  pos_weights.pt   [4.940222263336182, 10.0, 6.824522018432617, 5.746847629547119, 1.6120916604995728, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0, 10.0]
  settings.pkl
